# Split a trajectory into PCA x Graph subsets

Runs locally against a pip-installed `idp-pharmacophore-tools` and files already on disk -- no upload widget, no Google Colab dependency. Prefer to run this in Google Colab instead (no local install, and Colab's free compute is handy even though GPU specifically doesn't help this notebook -- see the note below)? Use `colab/01_split_trajectory_pca_graph.ipynb` instead -- same pipeline, only the Setup and file-input/output cells differ.

Companion notebook for *Dynamic Pharmacophore Mapping for Intrinsically Disordered Drug Targets* ([paper](#) · [SI](#)).

Point this at one full protein-ligand trajectory + topology on disk. This notebook:
1. Runs PCA on the all-heavy-atom protein-ligand contact-distance matrix and K-means-clusters the trajectory into conformational states (**PCA clusters**).
2. Within each PCA cluster, builds a per-frame residue contact-network graph and clusters frames by Jaccard distance between those networks (**graph sub-clusters**).
3. Saves a trajectory (`.xtc`) + representative structure (`.gro`) for the full PCA cluster and for each graph sub-cluster, following this project's `<name>_PCA_C<i>[_Graph<j>]` naming convention.

Feed any of the resulting subsets into **`02_pharmacophore_from_subset.ipynb`** (this directory's local version, or `colab/02_pharmacophore_from_subset.ipynb` for Colab) to compute its pharmacophore maps.

Mirrors `analysis.ipynb` Sections 3 and 6c from the pipeline's development repo — no new algorithms here, just PCA + graph clustering wired up for a single trajectory.


**Note:** PCA (`compute_pca_trajectory`, sklearn + deeptime) and graph sub-clustering (`compute_graph_clustering`/`cluster_graph_clustering`, networkx + sklearn) are CPU-only here -- no CUDA/PyTorch path for either. (GPU *does* help in `02_pharmacophore_from_subset.ipynb` -- its voxel-grid steps auto-detect CUDA via PyTorch, if you have a local GPU.)


## 1. Setup

In [ ]:
%%capture
# Installs the full, correct dependency set automatically via idp-pharmacophore-tools' own
# pyproject.toml (unlike the Colab notebook's install cell, which deliberately skips packages
# Colab ships preinstalled -- a local env has none of those, so install the real package instead).
#
# Pick ONE of the following depending on what's actually been released so far (see PUBLISHING.md
# in the idp-pharmacophore-tools repo):
%pip install -q git+https://github.com/emalacs/idp-pharmacophore-tools.git
# %pip install -q --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ idp-pharmacophore-tools
# %pip install -q idp-pharmacophore-tools


In [ ]:
import idp_pharmacophore_tools as pu


## 2. Set input files


In [ ]:
TOPOLOGY   = '/path/to/system.gro'    # or .pdb
TRAJECTORY = '/path/to/traj.xtc'      # or .dcd

print(f'Topology  : {TOPOLOGY}')
print(f'Trajectory: {TRAJECTORY}')


In [ ]:
# Nothing else needed here -- TOPOLOGY/TRAJECTORY above already point directly at files on this
# machine. (The Colab notebook has a Google Drive alternative in this spot instead, since Colab
# can't see your local filesystem.)


## 3. Settings

In [ ]:
SIM_NAME       = 'my_trajectory'   # used as the output folder / subset name prefix
LIGAND_RESNAME = None              # None -> auto-detect from topology
STRIDE         = 1
OFFSET         = 0
OUTPUT_DIR     = f'./output/{SIM_NAME}'

# -- PCA parameters -----------------------------------------------------------
PCA_CONTACT_CUTOFF  = 0.6    # nm  -- all-atom distance cutoff for the feature matrix
PCA_GAUSSIAN_KERNEL = True   # apply Gaussian kernel exp(-d**2 / 2*sigma**2) before PCA
PCA_GAUSSIAN_SIGMA  = 0.7    # nm  -- kernel width (7 A); smaller = sharper contact boundary
PCA_DIM             = 10     # total principal components to compute
PCA_ANALYSIS_DIM    = 2      # PC dimensions used for K-means (must be <= PCA_DIM)
PCA_N_CLUSTERS      = 3      # number of trajectory (PCA) clusters
PCA_STANDARD_SCALER = True   # StandardScaler per feature after the Gaussian kernel

# -- Graph sub-clustering parameters, applied within each PCA cluster --------
GRAPH_CONTACT_CUTOFF         = 0.6    # nm -- residue-residue closest-atom contact cutoff for graph edges
GRAPH_USE_LIGAND             = False  # include the ligand as a residue in the contact graph?
GRAPH_CLUSTERING_METHOD      = 'agglomerative'   # or 'hdbscan'
GRAPH_N_CLUSTERS             = None   # set an int to use a fixed cluster count instead of the threshold below
GRAPH_LINKAGE                = 'average'
GRAPH_DISTANCE_THRESHOLD     = 0.4    # only used when GRAPH_N_CLUSTERS is None
GRAPH_MIN_POPULATION_FRACTION = 0.01  # drop graph sub-clusters below this fraction of the full trajectory

## 4. Inspect topology (optional)

In [ ]:
topo_info = pu.analyze_topology(TOPOLOGY)
for key, value in topo_info.items():
    print(f'{key}: {value}')

## 5. Load trajectory

In [ ]:
sim = pu.PharmacophoreTrajectory(TOPOLOGY, TRAJECTORY)
sim.load(ligand_resname=LIGAND_RESNAME, stride=STRIDE, offset=OFFSET)

## 6. PCA -- trajectory clustering
Builds the all-heavy-atom protein-ligand distance matrix (ligand atoms averaged), applies a Gaussian kernel, and runs PCA + K-means to identify conformational clusters. Frame indices per cluster are stored in `sim.pca_frames_cl`.

In [ ]:
# save_pca=True keeps the (n_frames x n_prot_heavy) distance DataFrame in sim.all_atom_distances_df
# for use by compute_pca_trajectory below.
sim.compute_all_atom_contacts(cutoff=PCA_CONTACT_CUTOFF, save_pca=True)

In [ ]:
projection, eigenvalues, eigenvectors = sim.compute_pca_trajectory(
    pca_dim         = PCA_DIM,
    analysis_dim    = PCA_ANALYSIS_DIM,
    n_clusters      = PCA_N_CLUSTERS,
    gaussian_kernel = PCA_GAUSSIAN_KERNEL,
    gaussian_sigma  = PCA_GAUSSIAN_SIGMA,
    standard_scaler = PCA_STANDARD_SCALER,
)

total = eigenvalues.sum()
for i, ev in enumerate(eigenvalues):
    print(f'PC{i+1}: {ev/total*100:.1f}% variance  (eigenvalue={ev:.4f})')

In [ ]:
import plotly.graph_objects as go

fes = sim.pca_fes_df
fig_fes = go.Figure(go.Heatmap(
    z=fes.values,
    x=[round(float(v), 3) for v in fes.columns],
    y=[round(float(v), 3) for v in fes.index],
    colorscale='Jet',
    colorbar=dict(title='Free Energy (kT)'),
))
fig_fes.update_layout(title='2D Free Energy Surface', xaxis_title='PC1', yaxis_title='PC2',
                      width=600, height=500)
fig_fes.show()

In [ ]:
if sim.pca_clusters_fig is not None:
    sim.pca_clusters_fig.show()

print(f'Silhouette score: {sim.pca_silhouette_score:.3f}')
print()
print(sim.pca_cluster_populations.to_string(index=False))

In [ ]:
# Free the PCA feature matrix -- cluster assignments are already stored in sim.pca_frames_cl.
sim.all_atom_distances_df   = None
sim.all_atom_contact_frames = None
import gc; gc.collect()
print('PCA feature matrix freed.')

## 7. Save PCA + graph subsets
For each PCA cluster: saves the full cluster trajectory (representative frame = the frame closest, in PC1..PCA_ANALYSIS_DIM space, to that cluster's K-means centroid), then runs graph sub-clustering within it and saves each graph sub-cluster trajectory.

Output layout, matching this project's `output/<subset>/` convention used by the web viewer:
```
{OUTPUT_DIR}/{SIM_NAME}_PCA_C<i>/ligand_centroid.pdb, cluster_trajectory.xtc, cluster_trajectory.gro
{OUTPUT_DIR}/{SIM_NAME}_PCA_C<i>_Graph<j>/ligand_centroid.pdb, cluster_trajectory.xtc, cluster_trajectory.gro
```

In [ ]:
import copy, os, shutil
import numpy as np

subset_paths = {}  # subset name -> {'trajectory': ..., 'structure': ...}

for cluster_idx in range(PCA_N_CLUSTERS):
    frames = sim.pca_frames_cl[cluster_idx]
    pca_name = f'{SIM_NAME}_PCA_C{cluster_idx}'
    pca_dir  = f'{OUTPUT_DIR}/{pca_name}'
    os.makedirs(pca_dir, exist_ok=True)

    print(f"\n{'='*60}\nPCA cluster {cluster_idx}: {len(frames)} frames "
          f"({100*len(frames)/sim.protein_traj.n_frames:.1f}% of trajectory)\n{'='*60}")

    # -- Slice into a cluster-local copy, same pattern as analysis.ipynb Section 6 --------------
    sim_cl = copy.copy(sim)
    sim_cl.prot_lig_traj    = sim.prot_lig_traj[frames]
    sim_cl.protein_traj     = sim.protein_traj[frames]
    sim_cl.ligand_traj      = sim.ligand_traj[frames]
    sim_cl.simulation_times = [sim.simulation_times[f] for f in frames]

    # -- Save the plain PCA-cluster trajectory -------------------------------------------------
    # Representative frame = the one closest to this cluster's K-means centroid, in the same
    # projection[:, :PCA_ANALYSIS_DIM] space compute_pca_trajectory() clustered on.
    cluster_points = projection[frames, :PCA_ANALYSIS_DIM]
    centroid       = sim.pca_cluster_centers[cluster_idx]
    local_rep      = int(np.argmin(np.linalg.norm(cluster_points - centroid, axis=1)))

    pca_traj_path = f'{pca_dir}/cluster_trajectory.xtc'
    pca_gro_path  = f'{pca_dir}/cluster_trajectory.gro'
    sim.prot_lig_traj[frames].save_xtc(pca_traj_path)
    sim.prot_lig_traj[frames[local_rep]].save_gro(pca_gro_path)
    subset_paths[pca_name] = {'trajectory': pca_traj_path, 'structure': pca_gro_path}
    print(f'Saved {pca_name}: {pca_traj_path}')

    # -- Graph sub-clustering within this PCA cluster ------------------------------------------
    sim_cl.compute_graph_clustering(
        cutoff=GRAPH_CONTACT_CUTOFF, use_ligand=GRAPH_USE_LIGAND,
        full_trajectory_frames=sim.protein_traj.n_frames,   # floor below is relative to the FULL trajectory
    )
    graph_df, graph_reps, graph_silhouette = sim_cl.cluster_graph_clustering(
        clustering_method=GRAPH_CLUSTERING_METHOD,
        n_clusters=GRAPH_N_CLUSTERS,
        linkage=GRAPH_LINKAGE,
        distance_threshold=GRAPH_DISTANCE_THRESHOLD,
    )
    print(f'Graph sub-clustering silhouette: {graph_silhouette}')
    print(graph_df['cluster'].value_counts())

    graph_saved = sim_cl.save_graph_cluster_trajectories(
        pca_dir, min_population_fraction=GRAPH_MIN_POPULATION_FRACTION,
    )

    # save_graph_cluster_trajectories() writes into {pca_dir}/graph_clusters/cluster_<rank>_trajectory.*
    # -- rename each rank into its own top-level {name}_PCA_C<i>_Graph<rank> subset folder, matching
    # the flat output/<subset>/ convention the rest of this project (and the web viewer) expects.
    for rank, info in enumerate(graph_saved.values()):
        graph_name = f'{pca_name}_Graph{rank}'
        graph_dir  = f'{OUTPUT_DIR}/{graph_name}'
        os.makedirs(graph_dir, exist_ok=True)
        graph_traj_path = f'{graph_dir}/cluster_trajectory.xtc'
        graph_gro_path  = f'{graph_dir}/cluster_trajectory.gro'
        shutil.move(info['trajectory_path'], graph_traj_path)
        shutil.move(info['structure_path'], graph_gro_path)
        subset_paths[graph_name] = {'trajectory': graph_traj_path, 'structure': graph_gro_path}
        print(f'Saved {graph_name}: {graph_traj_path}')
    shutil.rmtree(f'{pca_dir}/graph_clusters', ignore_errors=True)

    sim_cl.clear_results()
    del sim_cl

print(f'\n{len(subset_paths)} subsets saved under {OUTPUT_DIR}/')

## 8. Done
`OUTPUT_DIR` now holds each subset's `cluster_trajectory.xtc`/`cluster_trajectory.gro` pair directly on disk -- no zip/download step needed locally. Feed any of them into `02_pharmacophore_from_subset.ipynb` to compute its pharmacophore maps.


In [ ]:
import os

for root, _, files in os.walk(OUTPUT_DIR):
    for fname in sorted(files):
        print(os.path.relpath(os.path.join(root, fname), OUTPUT_DIR))
